In [1]:
from pathlib import Path
import os

from caf.base import DVector

os.chdir(r'..\..')
from land_use.norcom import apply_norcom, NorCOMResult
os.chdir(r'supporting_processes\norcom')

In [2]:
# Define path to forecast data
forecast_dir = Path(r"F:\Working\Land-Use\forecast_population_20250520\02_Final Outputs")
output_dir = forecast_dir / 'NorCOM'
output_dir.mkdir(parents=True, exist_ok=True)

In [3]:
# Define path to zonal data for application
zonal_data_dir = Path(r"I:\NorMITs NorCOM\Import\Zonal Data")

In [4]:
# Define path to NorCOM parameter estimation results (with subfolders of 0v1+ and 1v2+)
param_dir = Path(r"I:\NorMITs NorCOM\AECOM working\v38")

In [5]:
# 0v1+ Model Correction (applied relative to 0 cars category)
# 1v2+ Model Correction (applied relative to the 1 car category)
# using just UK wide derived average for scotland as there's no detailed census data to compare
adjustment_factors = {
    'NE': {"0_v_1+": -0.05, "1_v_2+": -0.265},
    'NW': {"0_v_1+": -0.075, "1_v_2+": -0.25},
    'YH': {"0_v_1+": -0.055, "1_v_2+": -0.245},
    'Wales': {"0_v_1+": -0.07, "1_v_2+": -0.3},
    'WM': {"0_v_1+": -0.08, "1_v_2+": -0.27},
    'EM': {"0_v_1+": -0.05, "1_v_2+": -0.25},
    'SW': {"0_v_1+": -0.07, "1_v_2+": -0.23},
    'EoE': {"0_v_1+": -0.065, "1_v_2+": -0.2},
    'Lon': {"0_v_1+": -0.015, "1_v_2+": -0.23},
    'SE': {"0_v_1+": -0.06, "1_v_2+": -0.2},
    'Scotland': {"0_v_1+": -0.1, "1_v_2+": -0.2},
}

In [6]:
# define household files
household_dvectors = list(forecast_dir.glob("Output Households_*.hdf"))

In [7]:
for file in household_dvectors:
    print(f"Processing household file {file}")
    
    # get region and year from file name (file names are assumed to be 'Output Households_EM_2033.hdf', so 'Output Households_{gor}_{year}.hdf')
    gor, year = file.with_suffix("").name.split("_")[-2], int(file.with_suffix("").name.split("_")[-1])
    print(f"GOR and year identified as {gor} and {year}")

    # get file name
    ref = file.name

    # get corresponding population file name
    pop_file = f"Output Pop_{gor}_{year}.hdf"
    print(f"Population file identified as {pop_file}")

    # load the NorCOM parameter results
    any_car_ownership = NorCOMResult.from_coefficients_csv(
        csv_path=param_dir / "0v1+" / "output" / "final_model_coefficients.csv",
        case_category='1+', noncase_category='0',
        zonal_lookups=zonal_data_dir / f"zonal_logit_data_v38_{year}.csv"
    )
    multiple_car_ownership = NorCOMResult.from_coefficients_csv(
        csv_path=param_dir / "1v2+" / "output" / "final_model_coefficients.csv",
        case_category='2+', noncase_category='1', dependent_category='1+',
        zonal_lookups=zonal_data_dir / f"zonal_logit_data_v38_{year}.csv"
    )

    # read in household data
    print(f"Loading {file}")
    input_households = DVector.load(file)

    # apply norcom
    print("Applying NorCOM")
    output_households = apply_norcom(
        any_car_ownership_result=any_car_ownership, 
        multiple_car_ownership_result=multiple_car_ownership,
        input_dvector=input_households,
        any_car_ownership_correction=adjustment_factors.get(gor).get("0_v_1+"),
        multiple_car_ownership_correction=adjustment_factors.get(gor).get("1_v_2+")
    )

    # save output households
    print(f"Saving output households to {output_dir / ref}")
    output_households.save(output_dir / ref)

    # read in forecast population (pre-norcom)
    print(f"Loading {file.parent / pop_file}")
    input_population = DVector.load(file.parent / pop_file)

    print("Applying NorCOM splits to population data")
    # get segmentations from the resulting households *except* car availability
    hh_segs = [
        seg for seg in output_households.segmentation.names if seg != "car_availability"
    ]

    # calculate factors of car available splits by zone and segment that have resulted from norcom application
    factors = output_households / output_households.aggregate(hh_segs)

    # get segmentations from the input population *except* car availability
    pop_segs = [
        seg for seg in input_population.segmentation.names if seg != "car_availability"
    ]

    # apply factors to population
    output_pop = input_population.aggregate(pop_segs) * factors

    # save output population (post-norcom)
    print(f"Saving output population to {output_dir / pop_file}")
    output_pop.save(output_dir / pop_file)

Processing household file F:\Working\Land-Use\forecast_population_20250520\02_Final Outputs\Output Households_EM_2033.hdf
GOR and year identified as EM and 2033
Population file identified as Output Pop_EM_2033.hdf


C:\Miniforge3\envs\normits_lu\Lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
C:\Miniforge3\envs\normits_lu\Lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


Loading F:\Working\Land-Use\forecast_population_20250520\02_Final Outputs\Output Households_EM_2033.hdf
Applying NorCOM


The input data started with 42,648 columns. Filtering to None results in 2,847 columns.
C:\Miniforge3\envs\normits_lu\Lib\site-packages\caf\base\data_structures.py:443: UserWarning: 0.0 dropped on seg validation.
  warnings.warn(f"{full_sum - cut_sum} dropped on seg validation.")
C:\Miniforge3\envs\normits_lu\Lib\site-packages\caf\base\data_structures.py:931: SegmentationWarning: This operation has changed the segmentation of the DVector from ['children', 'adults', 'ns_sec', 'adult_nssec', 'accom_h', 'total'] to ['adult_nssec', 'total', 'accom_h', 'adults', 'children', 'car_availability', 'ns_sec']. This can happen but it can also be a sign of an error. Check the output DVector.
  warnings.warn(
C:\Miniforge3\envs\normits_lu\Lib\site-packages\caf\base\segmentation.py:342: SegmentationWarning: Read in level car_availability is a subset of the segment. If this was not expected check the input segmentation.
  warnings.warn(
C:\Miniforge3\envs\normits_lu\Lib\site-packages\caf\base\segmenta

Saving output households to F:\Working\Land-Use\forecast_population_20250520\02_Final Outputs\NorCOM\Output Households_EM_2033.hdf
Loading F:\Working\Land-Use\forecast_population_20250520\02_Final Outputs\Output Pop_EM_2033.hdf
Applying NorCOM splits to population data


...

Loading F:\Working\Land-Use\forecast_population_20250520\02_Final Outputs\Output Households_YH_2053.hdf
Applying NorCOM


The input data started with 42,648 columns. Filtering to None results in 3,355 columns.
C:\Miniforge3\envs\normits_lu\Lib\site-packages\caf\base\data_structures.py:443: UserWarning: 0.0 dropped on seg validation.
  warnings.warn(f"{full_sum - cut_sum} dropped on seg validation.")
C:\Miniforge3\envs\normits_lu\Lib\site-packages\caf\base\data_structures.py:931: SegmentationWarning: This operation has changed the segmentation of the DVector from ['children', 'adults', 'ns_sec', 'adult_nssec', 'accom_h', 'total'] to ['adult_nssec', 'total', 'accom_h', 'adults', 'children', 'car_availability', 'ns_sec']. This can happen but it can also be a sign of an error. Check the output DVector.
  warnings.warn(
C:\Miniforge3\envs\normits_lu\Lib\site-packages\caf\base\segmentation.py:342: SegmentationWarning: Read in level car_availability is a subset of the segment. If this was not expected check the input segmentation.
  warnings.warn(
C:\Miniforge3\envs\normits_lu\Lib\site-packages\caf\base\segmenta

Saving output households to F:\Working\Land-Use\forecast_population_20250520\02_Final Outputs\NorCOM\Output Households_YH_2053.hdf
Loading F:\Working\Land-Use\forecast_population_20250520\02_Final Outputs\Output Pop_YH_2053.hdf
Applying NorCOM splits to population data


C:\Miniforge3\envs\normits_lu\Lib\site-packages\caf\base\data_structures.py:443: UserWarning: 0.0 dropped on seg validation.
  warnings.warn(f"{full_sum - cut_sum} dropped on seg validation.")
C:\Miniforge3\envs\normits_lu\Lib\site-packages\caf\base\data_structures.py:443: UserWarning: 0.0 dropped on seg validation.
  warnings.warn(f"{full_sum - cut_sum} dropped on seg validation.")
C:\Miniforge3\envs\normits_lu\Lib\site-packages\caf\base\data_structures.py:931: SegmentationWarning: This operation has changed the segmentation of the DVector from ['children', 'g', 'economic_status', 'soc', 'adult_nssec', 'accom_h', 'pop_emp', 'ns_sec', 'adults', 'status_aps', 'age_ntem'] to ['adult_nssec', 'total', 'age_ntem', 'g', 'economic_status', 'adults', 'ns_sec', 'status_aps', 'children', 'car_availability', 'soc', 'accom_h', 'pop_emp']. This can happen but it can also be a sign of an error. Check the output DVector.
  warnings.warn(


Saving output population to F:\Working\Land-Use\forecast_population_20250520\02_Final Outputs\NorCOM\Output Pop_YH_2053.hdf
